# Import tools
Simulation, Data Manipulation and Math Operations

In [1]:
# Importing libraries & frameworks 
import torch 
import genesis as gs
import numpy as np
import torch.nn.functional as F
import json
from pathlib import Path
from scipy.spatial.transform import Rotation as R

[I 05/15/25 12:40:08.996 117349] [shell.py:_shell_pop_print@23] Graphical python shell detected, using wrapped sys.stdout


# PyTorch3D

A nice librarie for 3D operations - because of some problems of the python environment (conda) it wasn't possible to import this library properly. So the source functions are pasted here below.

## Transforms Operation functions
Reference: https://pytorch3d.readthedocs.io/en/latest/modules/transforms.html

In [2]:
# from pytorch3d

def _axis_angle_rotation(axis: str, angle: torch.Tensor) -> torch.Tensor:
    """
    Return the rotation matrices for one of the rotations about an axis
    of which Euler angles describe, for each value of the angle given.

    Args:
        axis: Axis label "X" or "Y or "Z".
        angle: any shape tensor of Euler angles in radians

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """

    cos = torch.cos(angle)
    sin = torch.sin(angle)
    one = torch.ones_like(angle)
    zero = torch.zeros_like(angle)

    if axis == "X":
        R_flat = (one, zero, zero, zero, cos, -sin, zero, sin, cos)
    elif axis == "Y":
        R_flat = (cos, zero, sin, zero, one, zero, -sin, zero, cos)
    elif axis == "Z":
        R_flat = (cos, -sin, zero, sin, cos, zero, zero, zero, one)
    else:
        raise ValueError("letter must be either X, Y or Z.")

    return torch.stack(R_flat, -1).reshape(angle.shape + (3, 3))

def euler_angles_to_matrix(euler_angles: torch.Tensor, convention: str) -> torch.Tensor:
    """
    Convert rotations given as Euler angles in radians to rotation matrices.

    Args:
        euler_angles: Euler angles in radians as tensor of shape (..., 3).
        convention: Convention string of three uppercase letters from
            {"X", "Y", and "Z"}.

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    if euler_angles.dim() == 0 or euler_angles.shape[-1] != 3:
        raise ValueError("Invalid input euler angles.")
    if len(convention) != 3:
        raise ValueError("Convention must have 3 letters.")
    if convention[1] in (convention[0], convention[2]):
        raise ValueError(f"Invalid convention {convention}.")
    for letter in convention:
        if letter not in ("X", "Y", "Z"):
            raise ValueError(f"Invalid letter {letter} in convention string.")
    matrices = [
        _axis_angle_rotation(c, e)
        for c, e in zip(convention, torch.unbind(euler_angles, -1))
    ]
    # return functools.reduce(torch.matmul, matrices)
    return torch.matmul(torch.matmul(matrices[0], matrices[1]), matrices[2])

def standardize_quaternion(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert a unit quaternion to a standard form: one in which the real
    part is non negative.

    Args:
        quaternions: Quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Standardized quaternions as tensor of shape (..., 4).
    """
    return torch.where(quaternions[..., 0:1] < 0, -quaternions, quaternions)

def quaternion_raw_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Multiply two quaternions.
    Usual torch rules for broadcasting apply.

    Args:
        a: Quaternions as tensor of shape (..., 4), real part first.
        b: Quaternions as tensor of shape (..., 4), real part first.

    Returns:
        The product of a and b, a tensor of quaternions shape (..., 4).
    """
    aw, ax, ay, az = torch.unbind(a, -1)
    bw, bx, by, bz = torch.unbind(b, -1)
    ow = aw * bw - ax * bx - ay * by - az * bz
    ox = aw * bx + ax * bw + ay * bz - az * by
    oy = aw * by - ax * bz + ay * bw + az * bx
    oz = aw * bz + ax * by - ay * bx + az * bw
    return torch.stack((ow, ox, oy, oz), -1)

def quaternion_multiply(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """
    Multiply two quaternions representing rotations, returning the quaternion
    representing their composition, i.e. the versor with nonnegative real part.
    Usual torch rules for broadcasting apply.

    Args:
        a: Quaternions as tensor of shape (..., 4), real part first.
        b: Quaternions as tensor of shape (..., 4), real part first.

    Returns:
        The product of a and b, a tensor of quaternions of shape (..., 4).
    """
    ab = quaternion_raw_multiply(a, b)
    return standardize_quaternion(ab)

def _sqrt_positive_part(x: torch.Tensor) -> torch.Tensor:
    """
    Returns torch.sqrt(torch.max(0, x))
    but with a zero subgradient where x is 0.
    """
    ret = torch.zeros_like(x)
    positive_mask = x > 0
    if torch.is_grad_enabled():
        ret[positive_mask] = torch.sqrt(x[positive_mask])
    else:
        ret = torch.where(positive_mask, torch.sqrt(x), ret)
    return ret

def matrix_to_quaternion(matrix: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to quaternions.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    """
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")

    batch_dim = matrix.shape[:-2]
    m00, m01, m02, m10, m11, m12, m20, m21, m22 = torch.unbind(
        matrix.reshape(batch_dim + (9,)), dim=-1
    )

    q_abs = _sqrt_positive_part(
        torch.stack(
            [
                1.0 + m00 + m11 + m22,
                1.0 + m00 - m11 - m22,
                1.0 - m00 + m11 - m22,
                1.0 - m00 - m11 + m22,
            ],
            dim=-1,
        )
    )

    # we produce the desired quaternion multiplied by each of r, i, j, k
    quat_by_rijk = torch.stack(
        [
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([q_abs[..., 0] ** 2, m21 - m12, m02 - m20, m10 - m01], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m21 - m12, q_abs[..., 1] ** 2, m10 + m01, m02 + m20], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m02 - m20, m10 + m01, q_abs[..., 2] ** 2, m12 + m21], dim=-1),
            # pyre-fixme[58]: `**` is not supported for operand types `Tensor` and
            #  `int`.
            torch.stack([m10 - m01, m20 + m02, m21 + m12, q_abs[..., 3] ** 2], dim=-1),
        ],
        dim=-2,
    )

    # We floor here at 0.1 but the exact level is not important; if q_abs is small,
    # the candidate won't be picked.
    flr = torch.tensor(0.1).to(dtype=q_abs.dtype, device=q_abs.device)
    quat_candidates = quat_by_rijk / (2.0 * q_abs[..., None].max(flr))

    # if not for numerical problems, quat_candidates[i] should be same (up to a sign),
    # forall i; we pick the best-conditioned one (with the largest denominator)
    out = quat_candidates[
        F.one_hot(q_abs.argmax(dim=-1), num_classes=4) > 0.5, :
    ].reshape(batch_dim + (4,))
    return standardize_quaternion(out)

def axis_angle_to_quaternion(axis_angle: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as axis/angle to quaternions.

    Args:
        axis_angle: Rotations given as a vector in axis angle form,
            as a tensor of shape (..., 3), where the magnitude is
            the angle turned anticlockwise in radians around the
            vector's direction.

    Returns:
        quaternions with real part first, as tensor of shape (..., 4).
    """
    angles = torch.norm(axis_angle, p=2, dim=-1, keepdim=True)
    sin_half_angles_over_angles = 0.5 * torch.sinc(angles * 0.5 / torch.pi)
    return torch.cat(
        [torch.cos(angles * 0.5), axis_angle * sin_half_angles_over_angles], dim=-1
    )

def vectors_to_quaternion(vectors):
    """
    Convert a batch of 3D vectors to Quaternion angles (wxyz convention).
    
    Parameters:
    vectors : np.ndarray, shape (n, 3)
        Array of n 3D vectors.
    
    Returns:
    quaternions : np.ndarray, shape (n, 4)
        Array of quaternions for each input vector.
    """
    # Ensure input is a PyTorch tensor with shape (n, 3)
    vectors = torch.as_tensor(vectors, dtype=torch.float32)
    if vectors.ndim != 2 or vectors.shape[1] != 3:
        raise ValueError("Input must be a tensor of shape (n, 3)")
    
    # Normalize the vectors
    norms = torch.norm(vectors, dim=1, keepdim=True)  # (n, 1)
    if torch.any(norms == 0):
        raise ValueError("Input contains zero vectors")
    normalized_vectors = vectors / norms
    
    # Reference vector
    reference = torch.tensor([1.0, 0.0, 0.0])
    
    # Compute rotation axes and angles
    dots = torch.clamp(torch.sum(normalized_vectors * reference, dim=1), -1.0, 1.0)
    angles = torch.arccos(dots)
    
    axes = torch.cross(reference.expand_as(normalized_vectors), normalized_vectors)
    axis_norms = torch.norm(axes, dim=1, keepdim=True)
    axes = axes / (axis_norms + 1e-10)  # Avoid division by zero
        
    # Compute rotation vectors (axis-angle)
    rotvecs = angles[:, None] * axes
        
    # Convert to quaternions using provided function
    quaternions = axis_angle_to_quaternion(rotvecs)

    # # Handle cases where vectors are parallel to reference
    # axis_norms = np.linalg.norm(axes, axis=1) # (n, )
    # near_zero = axis_norms < 1e-6
    # axes[near_zero] = np.array([0.0, 0.0, 1.0])  # Arbitrary perpendicular axis
    # axis_norms[near_zero] = 1.0
    
    # # Normalize rotation axes
    # axes = axes / axis_norms[:, np.newaxis] # (n, 3)  / (n, 1)
    
    # # Compute rotation vectors
    # rotvecs = angles[:, np.newaxis] * axes  # (n, 1) * (n, 3)
    
    # # Convert to quaternion angles
    # quaternions = axis_angle_to_quaternion(rotvecs)
    
    # Handle vectors aligned with reference (return zero Euler angles)
    # aligned = np.allclose(normalized_vectors, reference, atol=1e-6)
    # quaternions[aligned] = torch.tensor([1.0, 0., 0., 0.])
    
    return quaternions

def vectors_to_euler(vectors):
    """
    Convert a batch of 3D vectors to Euler angles (xyz convention).
    
    Parameters:
    vectors : np.ndarray, shape (n, 3)
        Array of n 3D vectors.
    
    Returns:
    euler_angles : np.ndarray, shape (n, 3)
        Array of Euler angles in radians for each input vector.
    """
    # Ensure input is a numpy array with shape (n, 3)
    vectors = np.asarray(vectors)
    if vectors.ndim != 2 or vectors.shape[1] != 3:
        raise ValueError("Input must be an array of shape (n, 3)")
    
    # Normalize the vectors
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) # (n, 3) -> (n, 1)
    if np.any(norms == 0):
        raise ValueError("Input contains zero vectors")
    normalized_vectors = vectors / norms
    
    # Reference vector
    reference = np.array([1.0, 0.0, 0.0])
    
    # Compute rotation axes and angles
    axes = np.cross(reference, normalized_vectors)
    dots = np.dot(normalized_vectors, reference)
    angles = np.arccos(dots)
    
    # Handle cases where vectors are parallel to reference
    axis_norms = np.linalg.norm(axes, axis=1) # (n, )
    near_zero = axis_norms < 1e-6
    axes[near_zero] = np.array([0.0, 0.0, 1.0])  # Arbitrary perpendicular axis
    axis_norms[near_zero] = 1.0
    
    # Normalize rotation axes
    axes = axes / axis_norms[:, np.newaxis] # (n, 3)  / (n, 1)
    
    # Compute rotation vectors
    rotvecs = angles[:, np.newaxis] * axes  # (n, 1) * (n, 3)
    
    # Convert to Euler angles
    rotation = R.from_rotvec(rotvecs)
    euler_angles = rotation.as_euler('xyz')
    
    # Handle vectors aligned with reference (return zero Euler angles)
    aligned = np.allclose(normalized_vectors, reference, atol=1e-6)
    euler_angles[aligned] = 0.0
    
    return euler_angles

def quaternion_to_axis_angle(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as quaternions to axis/angle.

    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Rotations given as a vector in axis angle form, as a tensor
            of shape (..., 3), where the magnitude is the angle
            turned anticlockwise in radians around the vector's
            direction.
    """
    norms = torch.norm(quaternions[..., 1:], p=2, dim=-1, keepdim=True)
    half_angles = torch.atan2(norms, quaternions[..., :1])
    sin_half_angles_over_angles = 0.5 * torch.sinc(half_angles / torch.pi)
    # angles/2 are between [-pi/2, pi/2], thus sin_half_angles_over_angles
    # can't be zero
    return quaternions[..., 1:] / sin_half_angles_over_angles

def quaternion_to_matrix(quaternions: torch.Tensor) -> torch.Tensor:
    """
    Convert rotations given as quaternions to rotation matrices.

    Args:
        quaternions: quaternions with real part first,
            as tensor of shape (..., 4).

    Returns:
        Rotation matrices as tensor of shape (..., 3, 3).
    """
    r, i, j, k = torch.unbind(quaternions, -1)
    # pyre-fixme[58]: `/` is not supported for operand types `float` and `Tensor`.
    two_s = 2.0 / (quaternions * quaternions).sum(-1)

    o = torch.stack(
        (
            1 - two_s * (j * j + k * k),
            two_s * (i * j - k * r),
            two_s * (i * k + j * r),
            two_s * (i * j + k * r),
            1 - two_s * (i * i + k * k),
            two_s * (j * k - i * r),
            two_s * (i * k - j * r),
            two_s * (j * k + i * r),
            1 - two_s * (i * i + j * j),
        ),
        -1,
    )
    return o.reshape(quaternions.shape[:-1] + (3, 3))

def matrix_to_euler_angles(matrix: torch.Tensor, convention: str) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to Euler angles in radians.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).
        convention: Convention string of three uppercase letters.

    Returns:
        Euler angles in radians as tensor of shape (..., 3).
    """
    if len(convention) != 3:
        raise ValueError("Convention must have 3 letters.")
    if convention[1] in (convention[0], convention[2]):
        raise ValueError(f"Invalid convention {convention}.")
    for letter in convention:
        if letter not in ("X", "Y", "Z"):
            raise ValueError(f"Invalid letter {letter} in convention string.")
    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")
    i0 = _index_from_letter(convention[0])
    i2 = _index_from_letter(convention[2])
    tait_bryan = i0 != i2
    if tait_bryan:
        central_angle = torch.asin(
            matrix[..., i0, i2] * (-1.0 if i0 - i2 in [-1, 2] else 1.0)
        )
    else:
        central_angle = torch.acos(matrix[..., i0, i0])

    o = (
        _angle_from_tan(
            convention[0], convention[1], matrix[..., i2], False, tait_bryan
        ),
        central_angle,
        _angle_from_tan(
            convention[2], convention[1], matrix[..., i0, :], True, tait_bryan
        ),
    )
    return torch.stack(o, -1)

def _index_from_letter(letter: str) -> int:
    if letter == "X":
        return 0
    if letter == "Y":
        return 1
    if letter == "Z":
        return 2
    raise ValueError("letter must be either X, Y or Z.")

def _angle_from_tan(
    axis: str, other_axis: str, data, horizontal: bool, tait_bryan: bool
) -> torch.Tensor:
    """
    Extract the first or third Euler angle from the two members of
    the matrix which are positive constant times its sine and cosine.

    Args:
        axis: Axis label "X" or "Y or "Z" for the angle we are finding.
        other_axis: Axis label "X" or "Y or "Z" for the middle axis in the
            convention.
        data: Rotation matrices as tensor of shape (..., 3, 3).
        horizontal: Whether we are looking for the angle for the third axis,
            which means the relevant entries are in the same row of the
            rotation matrix. If not, they are in the same column.
        tait_bryan: Whether the first and third axes in the convention differ.

    Returns:
        Euler Angles in radians for each matrix in data as a tensor
        of shape (...).
    """

    i1, i2 = {"X": (2, 1), "Y": (0, 2), "Z": (1, 0)}[axis]
    if horizontal:
        i2, i1 = i1, i2
    even = (axis + other_axis) in ["XY", "YZ", "ZX"]
    if horizontal == even:
        return torch.atan2(data[..., i1], data[..., i2])
    if tait_bryan:
        return torch.atan2(-data[..., i2], data[..., i1])
    return torch.atan2(data[..., i2], -data[..., i1])

def matrix_to_axis_angle(matrix: torch.Tensor, fast: bool = False) -> torch.Tensor:
    """
    Convert rotations given as rotation matrices to axis/angle.

    Args:
        matrix: Rotation matrices as tensor of shape (..., 3, 3).
        fast: Whether to use the new faster implementation (based on the
            Rodrigues formula) instead of the original implementation (which
            first converted to a quaternion and then back to a rotation matrix).

    Returns:
        Rotations given as a vector in axis angle form, as a tensor
            of shape (..., 3), where the magnitude is the angle
            turned anticlockwise in radians around the vector's
            direction.

    """
    if not fast:
        return quaternion_to_axis_angle(matrix_to_quaternion(matrix))

    if matrix.size(-1) != 3 or matrix.size(-2) != 3:
        raise ValueError(f"Invalid rotation matrix shape {matrix.shape}.")

    omegas = torch.stack(
        [
            matrix[..., 2, 1] - matrix[..., 1, 2],
            matrix[..., 0, 2] - matrix[..., 2, 0],
            matrix[..., 1, 0] - matrix[..., 0, 1],
        ],
        dim=-1,
    )
    norms = torch.norm(omegas, p=2, dim=-1, keepdim=True)
    traces = torch.diagonal(matrix, dim1=-2, dim2=-1).sum(-1).unsqueeze(-1)
    angles = torch.atan2(norms, traces - 1)

    zeros = torch.zeros(3, dtype=matrix.dtype, device=matrix.device)
    omegas = torch.where(torch.isclose(angles, torch.zeros_like(angles)), zeros, omegas)

    near_pi = angles.isclose(angles.new_full((1,), torch.pi)).squeeze(-1)

    axis_angles = torch.empty_like(omegas)
    axis_angles[~near_pi] = (
        0.5 * omegas[~near_pi] / torch.sinc(angles[~near_pi] / torch.pi)
    )

    # this derives from: nnT = (R + 1) / 2
    n = 0.5 * (
        matrix[near_pi][..., 0, :]
        + torch.eye(1, 3, dtype=matrix.dtype, device=matrix.device)
    )
    axis_angles[near_pi] = angles[near_pi] * n / torch.norm(n)

    return axis_angles



# Initializing Genesis

Below is defined some setup the scene and camera as well as the definition of the Rigid Body Entities used in the Grasp (Bottle and the Spot Gripper). 

Note: in the last line is defined the number of Parallel Environment.

In [3]:
# Genesis initialization 
gs.init(
    backend=gs.cuda,
    seed=None,
    debug=False,
    logging_level=None,
    theme='dark',
    logger_verbose_time=False
)

# Creating the scene
scene = gs.Scene(
    viewer_options=gs.options.ViewerOptions(
        camera_pos=(0, 0, 1.0),
        camera_lookat=(0, 0, 0.15),
        camera_fov=30,
        max_FPS=600
    ),
    vis_options=gs.options.VisOptions(show_world_frame=False),
    show_viewer=False, 
    show_FPS=False
)

# Plane (ground) 
plane = scene.add_entity(gs.morphs.Plane())

bottle_radius = 0.025
bottle_pos = [0, 0.0, 0.1] 

plastic_bottle = scene.add_entity(
    gs.morphs.Mesh(
        file="../bottles/plastic_bottle/bottle.stl", 
        pos=bottle_pos,            
        euler=(0, 0, 0.0),
        collision=True,
        visualization=True,
        fixed=True,          
        scale=0.0002                      
    )
)
initial_gripper_pos = [-1, 0.0, 0.10]
initial_gripper_quat = [0.707, 0.707, 0, 0]
spot_gripper = scene.add_entity(
    gs.morphs.URDF(
        file='/home/nexus/Desktop/spot_gen/spot_arm_description/spot_arm/urdf/open_gripper.urdf',
        quat = initial_gripper_quat,
        pos=initial_gripper_pos,
        scale=1.0,
        merge_fixed_links=True,
        fixed = True
    ),
)

# Camera Setup
cam = scene.add_camera(
    pos    =(3, -1.5, 0.2),
    lookat = (0.0, 0.0, 0.09),
    res    = (640, 480),
    fov    = 30,
    GUI    = False,
    up = (0, 0, 1),

)
# Building the Scene
scene.build(n_envs= 29755) # 20_000 29755

[Genesis] [12:40:11] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [12:40:11] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [12:40:11] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [12:40:11] [INFO] Running on [NVIDIA RTX A2000 12GB] with backend gs.cuda. Device memory: 11.75 GB.
[Genesis] [12:40:11] [INFO] 🚀 Genesis initialized. 🔖 version: 0.2.1, 🌱 seed: None, 📏 precision: '32', 🐛 debug: False, 🎨 theme: 'dark'.
[Genesis] [12:40:11] [INFO] Scene <f633d5a> created.
[Genesis] [12:40:11] [INFO] Adding <gs.RigidEntity>. idx: 0, uid: <59375f4>, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [12:40:11] [INFO] Adding <gs.RigidEntity>. idx: 1, uid: <b83e428>, morph: <gs.morphs.Mesh(file='/home/nexus/Desktop/spot_gen/Exp_RigidEntity/bottles/plastic_bottle/bottle.stl')>, material: <gs.materials.Rigid>.


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


[Genesis] [12:40:12] [INFO] Adding <gs.RigidEntity>. idx: 2, uid: <995498f>, morph: <gs.morphs.URDF(file='/home/nexus/Desktop/spot_gen/spot_arm_description/spot_arm/urdf/open_gripper.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [12:40:12] [INFO] Building scene <f633d5a>...
[Genesis] [12:40:20] [INFO] Compiling simulation kernels...
[Genesis] [12:40:48] [INFO] Building visualizer...


## Control Parameters Definition
Here is defined the range of kv, kp, the Degress of Freedom as well it's force.

In [4]:
# Define DOFs
hand_dof = np.arange(2)
finger_dof = np.array([1]) 
initial_gripper_dof_state = spot_gripper.get_dofs_position()  # Shape: (n_envs, num_dofs)
initial_finger_dof_pos = initial_gripper_dof_state[:, finger_dof].squeeze()  # Shape: (n_envs,)
print("Initial shapes: initial_gripper_dof_state:", initial_gripper_dof_state.shape, "initial_finger_dof_pos:", initial_finger_dof_pos.shape)
# Set PD control parameters
spot_gripper.set_dofs_kp(
    np.array([100]*2)
    )
spot_gripper.set_dofs_kv(
    np.array([1]*2)
    )
spot_gripper.set_dofs_force_range(
    np.array([-100]*2),
    np.array([100]*2)
    )

Initial shapes: initial_gripper_dof_state: torch.Size([29755, 2]) initial_finger_dof_pos: torch.Size([29755])


## Grasping helper functions

During the study some functions were created to make the automated grasp possible.

In [ ]:
# Grasping helper functions
def pixel_to_world(x, y, depth, cam_pos, cam_lookat, fov, img_width, img_height, world_up):
    """
    Convert a batch of pixel coordinates and depths to world coordinates.

    Args:
        x: array of pixel x-coordinates (horizontal) [N]
        y: array of pixel y-coordinates (vertical) [N]
        depth: array of depth values at those pixels [N]
        cam_pos: camera position (x, y, z)
        cam_lookat: point camera is looking at (x, y, z)
        fov: field of view in degrees
        img_width: width of the image in pixels
        img_height: height of the image in pixels
        world_up: world up vector (x, y, z)

    Returns:
        world_coords: array of (x, y, z) in world coordinates [N, 3]
    """
    # Convert inputs to numpy arrays
    cam_pos = np.array(cam_pos)
    cam_lookat = np.array(cam_lookat)
    world_up = np.array(world_up)
    x = np.array(x)
    y = np.array(y)
    depth = np.array(depth)

    # Calculate camera direction vector
    cam_dir = cam_lookat - cam_pos
    cam_dir = cam_dir / np.linalg.norm(cam_dir)

    # Calculate camera up vector
    cam_right = np.cross(cam_dir, world_up)
    cam_right = cam_right / np.linalg.norm(cam_right)
    cam_up = np.cross(cam_right, cam_dir)

    # Convert FOV to radians
    fov_rad = np.deg2rad(fov)

    # Calculate direction vector in camera space
    aspect_ratio = img_width / img_height
    tan_half_fov = np.tan(fov_rad / 2.0)

    # Calculate pixel coordinates in normalized device coordinates (-1 to 1)
    ndc_x = (2.0 * x / (img_width - 1)) - 1.0
    ndc_y = 1.0 - (2.0 * y / (img_height - 1))  # Flip y-axis

    # Calculate camera space coordinates
    cam_space_x = ndc_x * tan_half_fov * aspect_ratio
    cam_space_y = ndc_y * tan_half_fov
    cam_space_z = -1.0 * np.ones_like(cam_space_x)  # Negative z is forward

    # Create direction vectors in camera space
    cam_space_dir = np.stack([cam_space_x, cam_space_y, cam_space_z], axis=-1)
    cam_space_dir = cam_space_dir / np.linalg.norm(cam_space_dir, axis=-1, keepdims=True)

    # Convert to world space direction
    world_dir = (cam_right * cam_space_dir[..., 0:1] +
                 cam_up * cam_space_dir[..., 1:2] +
                 -cam_dir * cam_space_dir[..., 2:3])
    world_dir = world_dir / np.linalg.norm(world_dir, axis=-1, keepdims=True)

    # Calculate world positions
    world_pos = cam_pos + world_dir * depth[..., None]

    return world_pos

def perform_grasp(num_steps=10, pause_steps=50, max_gripper_pos=torch.pi/2, envs_idx=None):
    """
    Perform grasp by closing the gripper in specified environments.

    Args:
        num_steps (int): Number of steps to close the gripper.
        pause_steps (int): Simulation steps per grasp step for stabilization.
        max_gripper_pos (float): Maximum gripper joint position (radians).
        envs_idx (np.ndarray, optional): Environment indices to process. If None, process all.

    Returns:
        None
    """
    # Ensure envs_idx is a NumPy array
    if envs_idx is None:
        envs_idx = np.arange(scene.n_envs)
    envs_idx = np.asarray(envs_idx)
    num_envs_active = len(envs_idx)

    # Get current joint positions
    # print("Perform - Get Position")
    current_qpos = spot_gripper.get_dofs_position()  # Shape: (n_envs, num_dofs)
    start_gripper_pos = current_qpos[:, finger_dof]  # Shape: (n_envs,)
    
    if start_gripper_pos[envs_idx].shape != (num_envs_active,):
        start_gripper_pos = start_gripper_pos[envs_idx].squeeze()  # Ensure (num_envs_active,)

    # print(f"Perform - gripping")
    # Compute gripper positions for active environments
    next_gripper_pos = torch.full( (num_envs_active, ),  max_gripper_pos, device="cuda")
    next_gripper_pos = next_gripper_pos.squeeze()  # Ensure (num_envs_active,)
    
    # Update target positions for active environments
    target_qpos = current_qpos.clone()  # Shape: (n_envs, num_dofs)
    target_qpos[envs_idx, finger_dof] = next_gripper_pos  # Shape: (num_envs_active,)
    
    # Debug shapes
    # print(f"Step {i}: current_gripper_pos shape: {current_gripper_pos.shape}, target slice shape: {target_qpos[envs_idx, finger_dof].shape}")
    
    spot_gripper.control_dofs_position(target_qpos)
    
    # Run simulation steps
    for j in range(pause_steps):
        # print(f"Perform - {j}")
        scene.step()

    # print("Perform - Stabilization")
    for _ in range(20):
        scene.step()

def check_bottle_contact(envs_idx=None):
    """
    Check gripper-bottle contacts in specified environments using PyTorch.

    Args:
        envs_idx (np.ndarray, optional): Environment indices to process. If None, process all.

    Returns:
        torch.Tensor: Boolean tensor of shape (len(envs_idx),) indicating contact.
    """
    # Determine environments to process
    if envs_idx is None:
        envs_idx = np.arange(scene.n_envs)
    num_envs_active = len(envs_idx)
    envs_idx_tensor = torch.tensor(envs_idx, device='cuda', dtype=torch.long)

    # Get contact data
    contacts = spot_gripper.get_contacts(with_entity=plastic_bottle)
    
    # Handle empty or invalid contact data
    if not contacts or 'position' not in contacts:
        return torch.zeros(num_envs_active, dtype=torch.bool, device='cuda')

    # Initialize output tensor
    had_contact = torch.zeros(num_envs_active, dtype=torch.bool, device='cuda')
    
    # Get link names and find bottle link index
    link_names = [l.name for l in scene.rigid_solver.links]
    floor_link_idx = link_names.index("bottle_stl_baselink") if "bottle_stl_baselink" in link_names else -1
    if floor_link_idx == -1:
        raise ValueError("No bottle found on links")

    # Define gripper links and find their indices
    gripper_links = ["arm_link_wr1", "arm_link_fngr"]
    gripper_link_indices = torch.tensor(
        [link_names.index(name) for name in gripper_links if name in link_names],
        device='cuda',
        dtype=torch.long
    )
    if len(gripper_link_indices) == 0:
        return had_contact  # No gripper links found

    # Convert contact link indices to PyTorch tensors
    link_a = torch.tensor(contacts['link_a'], device='cuda', dtype=torch.long)
    link_b = torch.tensor(contacts['link_b'], device='cuda', dtype=torch.long)
    valid_mask = torch.tensor(contacts['valid_mask'], device='cuda', dtype=torch.bool)

    # Check if link_a or link_b are gripper links using torch.isin
    is_a_gripper = torch.isin(link_a, gripper_link_indices)  # (num_envs, num_contacts)
    is_b_gripper = torch.isin(link_b, gripper_link_indices)  # (num_envs, num_contacts)

    # Identify valid contacts involving gripper links
    valid_contacts = valid_mask & (is_a_gripper | is_b_gripper)  # (num_envs, num_contacts)

    # Filter for specified environments
    valid_contacts_active = valid_contacts[envs_idx_tensor]  # (num_envs_active, num_contacts)

    # Check if any contact in each environment is valid
    had_contact = valid_contacts_active.any(dim=1)  # (num_envs_active,)

    return had_contact

def check_floor_contact(envs_idx=None):
    """
    Check gripper-floor contacts in specified environments.

    Args:
        envs_idx (np.ndarray, optional): Environment indices to process. If None, process all.

    Returns:
        torch.Tensor: Boolean tensor of shape (len(envs_idx),) indicating contact.
    """
    if envs_idx is None:
        envs_idx = np.arange(scene.n_envs)
    num_envs_active = len(envs_idx)
    envs_idx_tensor = torch.tensor(envs_idx, device='cuda', dtype=torch.long)

    contacts = spot_gripper.get_contacts(with_entity=plane)
    
    if not contacts or 'position' not in contacts:
        return torch.zeros(num_envs_active, dtype=torch.bool, device='cuda')
    
    had_contact = torch.zeros(num_envs_active, dtype=torch.bool, device='cuda')
    
    link_names = [l.name for l in scene.rigid_solver.links]
    floor_link_idx = link_names.index("plane_baselink") if "plane_baselink" in link_names else -1

    if floor_link_idx == -1:
        raise ValueError("No floor found on links")

    # Convert valid_mask to PyTorch tensor
    valid_mask = torch.tensor(contacts['valid_mask'], device='cuda', dtype=torch.bool)

    # Use tensor operations for efficiency
    valid_contacts = valid_mask[envs_idx_tensor]  # (num_envs_active, num_contacts)
    had_contact = valid_contacts.any(dim=1)  # (num_envs_active,)

    return had_contact

def quaternion_from_euler(angles, seq='XYZ'):
    rot_matrix = euler_angles_to_matrix(angles, convention=seq.upper())
    quat_wxyz = matrix_to_quaternion(rot_matrix)  # Shape: (4,), [w, x, y, z]
    return quat_wxyz

def rotate_90(quaternion, envs_idx):
    converted_matrix_orientation = quaternion_to_matrix(quaternion)
    converted_euler_orientation = matrix_to_euler_angles(converted_matrix_orientation, convention="ZYX")
    converted_euler_orientation[:,2] = np.pi/2
    converted_matrix_from_euler = euler_angles_to_matrix(converted_euler_orientation, convention="ZYX")
    converted_quat_from_axis_angle = matrix_to_quaternion(converted_matrix_from_euler)
    spot_gripper.set_quat(converted_quat_from_axis_angle, envs_idx=envs_idx)

def approach(vector_normal, grasp_point, envs_idx, k_dist = 0.35):
    spot_gripper.set_pos((k_dist) * vector_normal+grasp_point, envs_idx=envs_idx)  


## Dataset Parameters

Below some parameters for the dataset creation are defined.

In [6]:
# Output directory
pre_dataset_dir = Path("../water_bottle_dataset")
pre_dataset_dir.mkdir(parents=True, exist_ok=True)

# Number of samples
num_samples = 1000

# Up direction reference
up = [0, 0, 1]

# Camera parameters
img_width, img_height, fov = 640, 480, 30

## Extraction Data Loop

In this loop all the randomization, robot action, data save happens.

In [7]:
from tqdm import tqdm
for i in tqdm(range(num_samples)):
    #  View setup 
    view_id = f"view_{i}" 
    view_dir = pre_dataset_dir / view_id

    # print("Load data")
    # Load saved data from .npy files
    rgb = np.load(view_dir / "rgb.npy")
    depth = np.load(view_dir / "depth.npy")
    normal_map = np.load(view_dir / "normal.npy")
    seg = np.load(view_dir / "segmentation.npy")
    bottle_pixels = np.where(seg == 1)
    bottle_pixels = np.transpose(np.array(bottle_pixels))  # Shape: (num_pixels, 2)
    # Load saved data from JSON file
    with open(view_dir / "metadata.json", "r") as f:
        metadata = json.load(f)
        cam_pos = metadata["camera_pos"]
        cam_lookat = metadata["camera_lookat"]
        up = metadata["up"]
        
    # Check number of bottle pixels
    num_pixels = len(bottle_pixels)
    if num_pixels == 0:
        print(f"Warning: No bottle pixels in {view_id}. Skipping grasp simulation.")
        continue

    # Limit number of active environments to number of bottle pixels
    active_envs = min(scene.n_envs, num_pixels)  # Use fewer grippers if num_pixels < n_envs
    active_env_indices = np.arange(active_envs)  # Indices: [0, 1, ..., active_envs-1]

    # Select random pixels (one per active environment)
    if num_pixels <= active_envs:
        # Use all pixels directly
        selected_indices = np.arange(num_pixels)
    else:
        # Randomly sample active_envs pixels
        selected_indices = np.random.choice(num_pixels, size=active_envs, replace=False)
    
    selected_pixels = bottle_pixels[selected_indices]  # Shape: (active_envs, 2)
    # print(f"Selected pixels: {selected_pixels.shape}")
    random_y, random_x = selected_pixels.T  # Shape: (active_envs,)

    # Get depth and normal at selected pixels
    specific_point_depth = depth[random_y, random_x]  # Shape: (active_envs,)
    specific_point_normal = normal_map[random_y, random_x]  # Shape: (active_envs, 3)
    vector_normal = 2 * (specific_point_normal - 127.5) / 255  # Normalize to [-1, 1]

    # Compute grasp points and orientations
    # print("Pixel to World")
    grasp_point = pixel_to_world(
        random_x, random_y, specific_point_depth, cam_pos, cam_lookat, fov, img_width, img_height, up
    )
    quaternions = vectors_to_quaternion(-torch.tensor(vector_normal))
    position = grasp_point + 0.5 * vector_normal.copy()

    # Set gripper poses only for active environments
    # print("Set pos/quat")
    spot_gripper.set_pos(position, envs_idx=active_env_indices)
    spot_gripper.set_quat(quaternions, envs_idx=active_env_indices)

    # Rotate gripper for better grasping
    # print("Rotate")
    rotate_90(quaternions, envs_idx=active_env_indices)
    
    # Move gripper closer to the bottle
    # print("Approach")
    approach(vector_normal, grasp_point, envs_idx=active_env_indices, k_dist=0.18)
    for _ in range(2):
        scene.step()

    # Perform grasp
    # print("finger_dof type:", type(finger_dof), "value:", finger_dof)
    # print("Perform")
    perform_grasp(envs_idx=active_env_indices)  # Modified to support envs_idx
    for _ in range(5):
        scene.step()
    # print("Check collision")
    # Check contacts only for active environments
    bottle_contacts = check_bottle_contact(envs_idx=active_env_indices)
    # print(f"\nGripper contacts with the bottle:\n{bottle_contacts}")
    floor_contacts = check_floor_contact(envs_idx=active_env_indices)
    # print(f"\nGripper contacts with the floor:\n{floor_contacts}")
    final_contacts = torch.logical_and(bottle_contacts, ~floor_contacts)
    # print(f"\nFinal contacts:\n{final_contacts}")
    # print("Save")
    final_contacts = final_contacts.cpu().numpy().tolist()  # Length: active_envs
    # Store metadata
    data_metadata = {
        "view_id": view_id,
        "camera_pos": cam_pos,
        "camera_lookat": cam_lookat,
        "up": up,
        "final_contacts": final_contacts,
        "random_y": random_y.tolist(),
        "random_x": random_x.tolist(),
    }

    # Save metadata
    with open(view_dir / "metadata.json", "w") as f:
        json.dump(data_metadata, f, indent=4)
    # print("Reset")
    # Reset grippers to initial location (all environments for stability)
    parallelized_pos = torch.tensor(initial_gripper_pos).repeat((scene.n_envs, 1))
    parallelized_quat = torch.tensor(initial_gripper_quat).repeat((scene.n_envs, 1))
    spot_gripper.set_pos(parallelized_pos)
    spot_gripper.set_quat(parallelized_quat)

    # Reset finger DOF positions for active environments
    current_qpos = spot_gripper.get_dofs_position()  # Shape: (n_envs, num_dofs)
    target_qpos = current_qpos.clone()  # Preserve current positions for other DOFs
    initial_dof_values = initial_finger_dof_pos[active_env_indices].squeeze()  # Shape: (active_envs,)
    target_qpos[active_env_indices, finger_dof] = initial_dof_values  # Shape: (active_envs,)

    # Debug print
    # print(f"finger_dof: {finger_dof}, type: {type(finger_dof)}")

    # Create dofs_idx_tensor
    dofs_idx_tensor = torch.tensor([
        finger_dof.item() if isinstance(finger_dof, (torch.Tensor, np.ndarray))
        else finger_dof[0] if isinstance(finger_dof, list)
        else finger_dof
    ], dtype=torch.int32)  # Ensure 1D tensor

    # Select only the DOF column corresponding to finger_dof for active environments
    target_dof_pos = target_qpos[active_env_indices, finger_dof]  # Shape: (active_envs,)
    target_dof_pos = target_dof_pos.unsqueeze(-1)  # Shape: (active_envs, 1) to match dofs_idx_tensor length

    # Debug print
    # print(f"Reset DOF: target_qpos shape: {target_qpos.shape}, target_dof_pos shape: {target_dof_pos.shape}, "
        # f"initial_dof_values shape: {initial_dof_values.shape}, dofs_idx_tensor: {dofs_idx_tensor}, "
        # f"dofs_idx_tensor shape: {dofs_idx_tensor.shape}")

    # Control the specific DOF position
    spot_gripper.set_dofs_position(target_dof_pos, dofs_idx_local=dofs_idx_tensor, envs_idx=active_env_indices)

    for _ in range(5):
        scene.step()
    print("Done")

  0%|          | 0/1000 [00:00<?, ?it/s]/tmp/ipykernel_117349/820296957.py:233: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:62.)
  axes = torch.cross(reference.expand_as(normalized_vectors), normalized_vectors)
[Genesis] [12:40:58] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:40:58] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:40:58] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:40:58] [WARNING] Base link is fixed. Overriding base link pose.


Perform - Get Position
Perform - gripping
Perform - 0
Perform - 1
Perform - 2
Perform - 3
Perform - 4
Perform - 5
Perform - 6
Perform - 7
Perform - 8
Perform - 9
Perform - 10
Perform - 11
Perform - 12
Perform - 13
Perform - 14
Perform - 15
Perform - 16
Perform - 17
Perform - 18
Perform - 19
Perform - 20
Perform - 21
Perform - 22
Perform - 23
Perform - 24
Perform - 25
Perform - 26
Perform - 27
Perform - 28
Perform - 29
Perform - 30
Perform - 31
Perform - 32
Perform - 33
Perform - 34
Perform - 35
Perform - 36
Perform - 37
Perform - 38
Perform - 39
Perform - 40
Perform - 41
Perform - 42
Perform - 43
Perform - 44
Perform - 45
Perform - 46
Perform - 47
Perform - 48
Perform - 49
Perform - Stabilization


[Genesis] [12:41:00] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:00] [WARNING] Base link is fixed. Overriding base link pose.
  0%|          | 1/1000 [00:02<44:16,  2.66s/it][Genesis] [12:41:00] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:00] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:00] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:00] [WARNING] Base link is fixed. Overriding base link pose.


Done
Perform - Get Position
Perform - gripping
Perform - 0
Perform - 1
Perform - 2
Perform - 3
Perform - 4
Perform - 5
Perform - 6
Perform - 7
Perform - 8
Perform - 9
Perform - 10
Perform - 11
Perform - 12
Perform - 13
Perform - 14
Perform - 15
Perform - 16
Perform - 17
Perform - 18
Perform - 19
Perform - 20
Perform - 21
Perform - 22
Perform - 23
Perform - 24
Perform - 25
Perform - 26
Perform - 27
Perform - 28
Perform - 29
Perform - 30
Perform - 31
Perform - 32
Perform - 33
Perform - 34
Perform - 35
Perform - 36
Perform - 37
Perform - 38
Perform - 39
Perform - 40
Perform - 41
Perform - 42
Perform - 43
Perform - 44
Perform - 45
Perform - 46
Perform - 47
Perform - 48
Perform - 49
Perform - Stabilization


[Genesis] [12:41:02] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:02] [WARNING] Base link is fixed. Overriding base link pose.
  0%|          | 2/1000 [00:04<37:26,  2.25s/it][Genesis] [12:41:02] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:02] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:02] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:41:02] [WARNING] Base link is fixed. Overriding base link pose.


...

[Genesis] [12:44:42] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:44:42] [WARNING] Base link is fixed. Overriding base link pose.
  7%|▋         | 69/1000 [03:44<52:58,  3.41s/it][Genesis] [12:44:42] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:44:42] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:44:42] [WARNING] Base link is fixed. Overriding base link pose.
[Genesis] [12:44:42] [WARNING] Base link is fixed. Overriding base link pose.


Done
Perform - Get Position


  7%|▋         | 69/1000 [03:44<50:32,  3.26s/it]

Perform - gripping
Perform - 0
Perform - 1
Perform - 2
Perform - 3
Perform - 4
Perform - 5
Perform - 6
Perform - 7
Perform - 8
Perform - 9
Perform - 10
Perform - 11
Perform - 12
Perform - 13
Perform - 14
Perform - 15
Perform - 16
Perform - 17
Perform - 18
Perform - 19
Perform - 20
Perform - 21
Perform - 22
Perform - 23


KeyboardInterrupt: 